# Practice 2 — Concurrency, Asyncio, and Resilience

asyncio, TaskGroups, timeouts, threads, retries, idempotency.

**How it works:** each exercise is almost-finished code with a few `___` blanks. Fill them in, run the cell, then run the `check(...)` cell below it. ✅ means move on. ❌ shows your result and the test input, never the answer, and failing twice unlocks a hint.

Prefer a terminal? The same exercises are in [`practice/exercises/advanced_02_concurrency_resilience.py`](../../practice/exercises/advanced_02_concurrency_resilience.py).

## Setup

Run this cell first.

In [ ]:
from __future__ import annotations
from check import Checker

check = Checker('advanced_02_concurrency_resilience')

import asyncio
import random
import threading
from collections.abc import Awaitable, Callable, Iterable

class TransientError(Exception):
    """A failure worth retrying (timeouts, 503s...)."""

class PermanentError(Exception):
    """A failure that will not fix itself (bad input, 404...)."""

class CircuitOpen(Exception):
    """Raised instead of calling the operation while the breaker is open."""

## Exercises

### 1. `gather_in_order`

Run worker(value) concurrently in an asyncio.TaskGroup; return results in INPUT order.

In [ ]:
async def gather_in_order(values: Iterable, worker: Callable[[object], Awaitable]) -> list:
    """Run worker(value) concurrently in an asyncio.TaskGroup; return results in INPUT order."""
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(worker(value)) for value in values]
    return ___

In [ ]:
check('gather_in_order')

### 2. `with_timeout`

Return ('ok', result) or ('timeout', None) using asyncio.timeout().

In [ ]:
async def with_timeout(awaitable: Awaitable, seconds: float) -> tuple[str, object]:
    """Return ('ok', result) or ('timeout', None) using asyncio.timeout()."""
    try:
        async with asyncio.timeout(seconds):
            return ___
    except TimeoutError:
        return ___

In [ ]:
check('with_timeout')

### 3. `run_bounded`

Run all workers concurrently but at most `limit` at a time (asyncio.Semaphore).

In [ ]:
async def run_bounded(values: Iterable, worker: Callable[[object], Awaitable], limit: int) -> tuple[list, int]:
    """Run all workers concurrently but at most `limit` at a time (asyncio.Semaphore).

    Return (results in input order, peak number of workers running at once).
    """
    semaphore = ___
    active = peak = 0

    async def guarded(value):
        nonlocal active, peak
        async with semaphore:
            active += 1
            peak = ___
            try:
                return ___
            finally:
                active -= 1

    results = ___
    return results, peak

In [ ]:
check('run_bounded')

### 4. `pipeline`

Producer/consumer through a bounded asyncio.Queue; a None sentinel stops the consumer.

In [ ]:
async def pipeline(items: Iterable, transform: Callable[[object], object], maxsize: int = 2) -> list:
    """Producer/consumer through a bounded asyncio.Queue; a None sentinel stops the consumer.

    Return transform(item) for each item, in order.
    """
    queue: asyncio.Queue = asyncio.Queue(maxsize=maxsize)
    output: list = []

    async def producer():
        for item in ___:
            await queue.put(item)
        await queue.put(None)

    async def consumer():
        while ___:
            output.append(transform(item))

    async with asyncio.TaskGroup() as group:
        group.create_task(___)
        group.create_task(___)
    return output

In [ ]:
check('pipeline')

### 5. `run_blocking`

Run a blocking function without blocking the event loop (asyncio.to_thread).

In [ ]:
async def run_blocking(func: Callable, *args):
    """Run a blocking function without blocking the event loop (asyncio.to_thread)."""
    return ___

In [ ]:
check('run_blocking')

### 6. `SafeCounter`

A counter that is safe to increment from many threads (threading.Lock).

In [ ]:
class SafeCounter:
    """A counter that is safe to increment from many threads (threading.Lock)."""

    def __init__(self) -> None:
        self._value = 0
        self._lock = ___

    def increment(self) -> None:
        with ___:
            self._value += 1

    @property
    def value(self) -> int:
        return ___

In [ ]:
check('SafeCounter')

### 7. `retry`

Call operation(); on TransientError wait base_delay * 2**n and try again.

In [ ]:
def retry(operation: Callable[[], object], attempts: int = 3, base_delay: float = 0.1,
          sleep: Callable[[float], None] = None) -> object:
    """Call operation(); on TransientError wait base_delay * 2**n and try again.

    PermanentError is re-raised immediately. After `attempts` failures re-raise the
    last TransientError. `sleep` is injected so tests never really wait.
    """
    import time
    sleep = sleep or time.sleep
    for attempt in ___:
        try:
            return ___
        except TransientError:
            if ___:
                raise
            sleep(base_delay * 2 ** attempt)
    raise ValueError("attempts must be at least 1")

In [ ]:
check('retry')

### 8. `jittered_delays`

Full-jitter backoff: delay n is uniform in [0, base * 2**n]. Same seed -> same delays.

In [ ]:
def jittered_delays(attempts: int, base: float, seed: int) -> list[float]:
    """Full-jitter backoff: delay n is uniform in [0, base * 2**n]. Same seed -> same delays."""
    rng = random.Random(seed)
    return ___

In [ ]:
check('jittered_delays')

### 9. `IdempotentPayments`

charge(key, amount) runs the side effect once per key; duplicates return the first result.

In [ ]:
class IdempotentPayments:
    """charge(key, amount) runs the side effect once per key; duplicates return the first result.

    Reusing a key with a DIFFERENT amount raises ValueError.
    """

    def __init__(self, charge_card: Callable[[float], str]):
        self._charge_card = ___
        self._results: dict[str, tuple[float, str]] = {}

    def charge(self, key: str, amount: float) -> str:
        if ___:
            previous_amount, receipt = self._results[key]
            if ___:
                raise ValueError("idempotency key reused with different parameters")
            return receipt
        receipt = self._charge_card(amount)
        self._results[key] = (amount, receipt)
        return receipt

In [ ]:
check('IdempotentPayments')

### 10. `first_result`

Run coroutines concurrently; return the first result to finish and cancel the rest.

In [ ]:
async def first_result(coroutines: list) -> object:
    """Run coroutines concurrently; return the first result to finish and cancel the rest.

    Hint: asyncio.wait(..., return_when=asyncio.FIRST_COMPLETED).
    """
    tasks = ___
    try:
        done, _ = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
        return ___
    finally:
        for task in tasks:
            task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)

In [ ]:
check('first_result')

### 11. `map_threaded`

Apply func to items with a ThreadPoolExecutor; results in input order.

In [ ]:
def map_threaded(func: Callable, items: list, workers: int = 4) -> list:
    """Apply func to items with a ThreadPoolExecutor; results in input order."""
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return ___

In [ ]:
check('map_threaded')

### 12. `TokenBucket`

Rate limiter: `capacity` tokens, refilled at `rate` tokens/second from an injected clock.

In [ ]:
class TokenBucket:
    """Rate limiter: `capacity` tokens, refilled at `rate` tokens/second from an injected clock.

    allow() consumes a token and returns True, or returns False if none are available.
    The bucket starts full and never holds more than capacity.
    """

    def __init__(self, capacity: int, rate: float, clock: Callable[[], float]):
        self.capacity, self.rate, self.clock = capacity, rate, clock
        self._tokens = ___
        self._last = clock()

    def allow(self) -> bool:
        now = ___
        self._tokens = min(self.capacity, self._tokens + (now - self._last) * self.rate)
        self._last = now
        if ___:
            self._tokens -= 1
            return True
        return False

In [ ]:
check('TokenBucket')

### 13. `CircuitBreaker`

After `threshold` consecutive failures the breaker opens: calls raise CircuitOpen

In [ ]:
class CircuitBreaker:
    """After `threshold` consecutive failures the breaker opens: calls raise CircuitOpen
    without running. After `reset_after` seconds (injected clock) one trial call is allowed;
    success closes the breaker, failure re-opens it. Success resets the failure count.

    state is 'closed', 'open' or 'half-open'.
    """

    def __init__(self, threshold: int, reset_after: float, clock: Callable[[], float]):
        self.threshold, self.reset_after, self.clock = ___
        self.failures = 0
        self._opened_at = None

    @property
    def state(self) -> str:
        if ___:
            return "closed"
        if ___:
            return "half-open"
        return "open"

    def call(self, operation: Callable[[], object]):
        if ___:
            raise CircuitOpen("circuit is open")
        try:
            result = ___
        except Exception:
            self.failures += 1
            if ___:
                self._opened_at = self.clock()
            raise
        self.failures = 0
        self._opened_at = None
        return result

In [ ]:
check('CircuitBreaker')

## Finished?

In [ ]:
check.summary()